# From one student to many · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 02. From one student to many

Represent several records with lists and dictionaries before building a table.

Before you start: Complete the dictionary examples in Module 1.

Official reference: https://docs.python.org/3/tutorial/datastructures.html

### M02C01. A list has positions

A list stores values in order. Positions start at 0, so names[1] is the second value. A slice such as [0:2] includes positions 0 and 1 but excludes position 2. A negative position counts back from the end.

**Read:** What do the square brackets mean when names is first created?

Answer: They create a list containing four text values in order.

**Predict:** What is printed by names[-1] and names[0:2]?

Answer: Dana, then ["Amira", "Bilal"].

**Explain:** Why does [0:2] return two names rather than three?

Answer: The start position is included, but the stop position 2 is excluded.



In [ ]:
# Predict before running: What is printed by names[-1] and names[0:2]?
names = ["Amira", "Bilal", "Chloe", "Dana"]
print(names[0])
print(names[-1])
result = names[0:2]
print(result)

**Your turn:** Select only Bilal and Chloe into result using one slice. Then append Ethan and print the new list length.

For the check, keep these result names: `names`, `result`.

**Hint:** Start the slice at position 1 and stop before position 3.

**Watch for this:** A single position returns one item; a slice returns a list, even if it contains one item.

In [ ]:
# Change this code to complete the task.
# Question: Select only Bilal and Chloe into result using one slice. Then append Ethan and print the new list length.
names = ["Amira", "Bilal", "Chloe", "Dana"]
result = names[1:3]
names.append("Ethan")
print(result)
print(len(names))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == ['Bilal', 'Chloe']
assert names[-1] == 'Ethan' and len(names) == 5
print("Code checks passed. Review the written explanation as well.")

The slice returns Bilal and Chloe. append() adds a value at the end, so the full list now has five names.

**Quick check:** Which expression selects the second value?

1. names[2]
2. names[1]
3. names[0]

Answer: 2. Positions start at 0, so the second value is at position 1.

**Explain what you learned:** I can predict list positions and slices.

### M02C02. A dictionary of lists describes columns

Now each dictionary value is a list. The keys describe future column names, and values at the same position belong to the same student. data["Age"] chooses a list; the second pair of brackets chooses an item inside that list.

**Read:** What is the type of data, and what is the type of data["Age"]?

Answer: data is a dictionary. data["Age"] is a list.

**Predict:** Which student is described by position 1 in all three lists?

Answer: Bilal, age 21, at Dubai.

**Explain:** What can go wrong if you reorder only the names list?

Answer: Names could become paired with another student's age and campus. The lists must stay aligned by position.



In [ ]:
# Predict before running: Which student is described by position 1 in all three lists?
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19],
    "Campus": ["RAK", "Dubai", "RAK"]
}
print(data["Name"])
print(data["Age"][1])
print(data["Name"][1], data["Campus"][1])

**Your turn:** Change Chloe's age from 19 to 20 without changing the other ages. Print Chloe's name and updated age.

For the check, keep these result names: `data`.

**Hint:** Chloe is at position 2 inside each list.

**Watch for this:** Positions in lists are numbers; dictionary keys can be text. Do not swap their roles.

In [ ]:
# Change this code to complete the task.
# Question: Change Chloe's age from 19 to 20 without changing the other ages. Print Chloe's name and updated age.
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19],
    "Campus": ["RAK", "Dubai", "RAK"]
}
data["Age"][2] = 20
print(data["Name"][2], data["Age"][2])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert data['Age'] == [20, 21, 20]
assert data['Name'][2] == 'Chloe'
print("Code checks passed. Review the written explanation as well.")

The age list becomes [20, 21, 20]. Updating one nested list position does not change the other positions.

**Quick check:** What does data["Age"][1] select?

1. The first key.
2. Every age.
3. The second age value.

Answer: 3. The first brackets retrieve the Age list; the second retrieve its position 1.

**Explain what you learned:** I can read a value from a dictionary containing lists.

### M02C03. A list of dictionaries describes rows

A second arrangement stores one dictionary per student inside a list. Each dictionary is a record, or row. rows[1] selects the second record; ["Name"] then selects a field within it. Both arrangements can later become a pandas DataFrame.

**Read:** What is stored in each element of rows?

Answer: One dictionary describing one student record.

**Predict:** What will rows[1]["Name"] print?

Answer: Bilal.

**Explain:** Why is the order of the brackets different from data["Name"][1] in the previous cell?

Answer: rows is organized as a list of records, so select the record first. data was organized as named columns, so select the column first.



In [ ]:
# Predict before running: What will rows[1]["Name"] print?
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21, "Campus": "Dubai"},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
print(rows[1])
print(rows[1]["Name"])
print(len(rows))

**Your turn:** Add Dana, age 22, at Dubai as a fourth dictionary. Print the last student's name and the total record count.

For the check, keep these result names: `rows`.

**Hint:** Use rows.append({...}) and then rows[-1]["Name"].

**Watch for this:** Count records with len(rows); len(rows[0]) counts fields in the first record.

In [ ]:
# Change this code to complete the task.
# Question: Add Dana, age 22, at Dubai as a fourth dictionary. Print the last student's name and the total record count.
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21, "Campus": "Dubai"},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
rows.append({"Name": "Dana", "Age": 22, "Campus": "Dubai"})
print(rows[-1]["Name"])
print(len(rows))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(rows) == 4 and rows[-1] == {'Name': 'Dana', 'Age': 22, 'Campus': 'Dubai'}
print("Code checks passed. Review the written explanation as well.")

Dana is the last record and there are four records. This arrangement keeps each student's fields together.

**Quick check:** Which arrangement stores one dictionary per row?

1. A list of dictionaries.
2. A dictionary of numbers.
3. A single text string.

Answer: 1. A list of dictionaries groups each record's fields inside its own dictionary.

**Explain what you learned:** I can tell column-oriented data from row-oriented records.

### M02C04. Check lengths before making a table

A rectangular table needs the same number of values in each plain list column. len() counts the values in a list. You can inspect all column lengths before constructing the DataFrame. A missing measurement still needs a position, often filled with None.

**Read:** What does == do here?

Answer: It compares two lengths and returns True or False. It does not assign a value.

**Predict:** What happens to same_length if you remove the last age only?

Answer: It becomes False: there would be three names but only two ages.

**Explain:** Why is adding a zero not always a good way to represent an unknown age?

Answer: Zero is a real numerical value. Using it could incorrectly suggest the student is zero years old and distort calculations.



In [ ]:
# Predict before running: What happens to same_length if you remove the last age only?
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19]
}
lengths = {"Name": len(data["Name"]), "Age": len(data["Age"])}
same_length = lengths["Name"] == lengths["Age"]
print(lengths)
print("Ready for a rectangular table:", same_length)

**Your turn:** Add Dana to the Name list. Her age is unknown, so append None to Age. Confirm that the two lengths still match.

For the check, keep these result names: `data`, `same_length`.

**Hint:** Append one value to each list; use None without quotes.

**Watch for this:** None is a missing-value marker here; "None" in quotes is ordinary text.

In [ ]:
# Change this code to complete the task.
# Question: Add Dana to the Name list. Her age is unknown, so append None to Age. Confirm that the two lengths still match.
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19]
}
data["Name"].append("Dana")
data["Age"].append(None)
same_length = len(data["Name"]) == len(data["Age"])
print(data)
print("Matching lengths:", same_length)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert same_length is True and len(data['Name']) == 4
assert data['Age'][-1] is None
print("Code checks passed. Review the written explanation as well.")

Both lists now contain four positions. None marks the missing age while preserving the row alignment.

**Quick check:** What is needed for a DataFrame built from plain list columns?

1. Every value must be text.
2. All column lists must have equal length.
3. Each column must have a different length.

Answer: 2. Plain list columns must have equal lengths. Indexed Series have separate alignment rules, introduced next.

**Explain what you learned:** I can check whether list-based columns form a consistent table.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates